# Intital EDA
before and after Bronze layer

In [0]:
VOLUME_PATH = "/Volumes/marathos/default/raw"

spark.sql(f"LIST '{VOLUME_PATH}'").display()

In [0]:
spark.sql(f"LIST '{VOLUME_PATH}/data'").display()

In [0]:
spark.sql(f"LIST '{VOLUME_PATH}/raw_data'").display()

In [0]:
BASE_DIR = "/Volumes/marathos/default/raw"

schema = (
    spark.read.format("csv")
    .options(header=True, inferSchema=True)
    .load(f"{BASE_DIR}/data/TWO_CENTURIES_OF_UM_RACES.csv")
    .schema
)

schema

# Bronze layer EDA

In [0]:
%sql

SELECT COUNT(*) FROM marathos.bronze.raw_marathos;

In [0]:
%sql

FROM marathos.bronze.raw_marathos LIMIT 10;

In [0]:
%sql

SELECT DISTINCT
    `Event name`
FROM marathos.bronze.raw_marathos

In [0]:
%sql

SELECT COUNT(DISTINCT `Event name`) AS total_unique_events
FROM marathos.bronze.raw_marathos

In [0]:
df = spark.table("marathos.bronze.raw_marathos")

df.display()

In [0]:
df.printSchema()


In [0]:
display(df.describe())

In [0]:
from pyspark.sql.functions import col, sum as spark_sum

null_counts = df.select(
    [spark_sum(col(column).isNull().cast("int")).alias(column) for column in df.columns]
)

null_counts = null_counts.collect()[0].asDict()
[(column, nulls)for column, nulls in null_counts.items() if nulls > 0]

In [0]:
import plotly.express as px

df_age_distribution = spark.sql("""
    SELECT age, COUNT(*) AS count
    FROM (
        SELECT `Year of event` - CAST(`Athlete year of birth` AS INT) AS age
        FROM marathos.bronze.raw_marathos
    )
    WHERE age > 10 AND age < 96
    GROUP BY age
    ORDER BY age
""")

pdf = df_age_distribution.toPandas()

fig = px.bar(pdf, x="age", y="count", title="Age Distribution of Runners")
fig.show()

In [0]:
df_country_count = spark.sql(
    """
    SELECT
        `Athlete country`,
        COUNT(`Athlete country`) AS runners_per_country
    FROM marathos.bronze.raw_marathos
    GROUP BY `Athlete country`
    ORDER BY runners_per_country DESC
    """
)

df_country_count.display()

In [0]:

df_country_count = df_country_count.limit(10)

pdf = df_country_count.toPandas()

fig = px.pie(pdf, names="Athlete country", values="runners_per_country", title="Top 10 Runners per country")
fig.show()